In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.inspection import permutation_importance
import joblib

In [2]:
DATASET_PATH = "../data/train.csv"

LOW_QUANTILE = 0.10
HIGH_QUANTILE = 0.90
IRQ_COEFF = 3

# Data Processing

### Loading Dataset

In [3]:
ds = pd.read_csv(DATASET_PATH)

important_df =  ds[['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'Color3', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'Fee', 'State', 'AdoptionSpeed']]

important_df = important_df.drop(columns=['Color3', 'Fee', 'State'])

In [4]:
important_df.head(5)

,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


In [5]:
missing = list()
for x in important_df.columns:
    if important_df[x].isnull().sum() != 0:
        print(f"{x:<30}{important_df[x].isnull().sum():<10}{(important_df[x].isnull().sum() / important_df.shape[0])*100}%")
        missing.append(x)

In [6]:
important_df.head(5)

,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


### Detecting anomalies

In [7]:
def detect_anomalies(df, column):
    Q1 = df[column].quantile(LOW_QUANTILE)
    Q3 = df[column].quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR
    anomalies = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return anomalies

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    anomalies = detect_anomalies(important_df, col)
    print(f"Anomalies in {col}:")
    print(anomalies.shape[0])

Anomalies in Age:
62
Anomalies in Breed1:
0
Anomalies in Breed2:
0
Anomalies in Gender:
0
Anomalies in Color1:
0
Anomalies in Color2:
0
Anomalies in MaturitySize:
0
Anomalies in Vaccinated:
0
Anomalies in FurLength:
0
Anomalies in Dewormed:
0
Anomalies in Sterilized:
0
Anomalies in Health:
515
Anomalies in Quantity:
62
Anomalies in AdoptionSpeed:
0


In [8]:
def replace_anomalies_with_minmax_values(df, column):
    sorted_values = df[column].sort_values()
    Q1 = sorted_values.quantile(LOW_QUANTILE)
    Q3 = sorted_values.quantile(HIGH_QUANTILE)
    IQR = Q3 - Q1
    lower_bound = Q1 - IRQ_COEFF * IQR
    upper_bound = Q3 + IRQ_COEFF * IQR

    min_real_value = sorted_values[sorted_values >= lower_bound].min()
    max_real_value = sorted_values[sorted_values <= upper_bound].max()

    df.loc[df[column] < lower_bound, column] = min_real_value
    df.loc[df[column] > upper_bound, column] = max_real_value

numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']

for col in numerical_columns:
    replace_anomalies_with_minmax_values(important_df, col)

print("Anomalies replaced with real values. Updated DataFrame:")
important_df.head(5)

Anomalies replaced with real values. Updated DataFrame:


,Age,Breed1,Breed2,Gender,Color1,Color2,MaturitySize,Vaccinated,FurLength,Dewormed,Sterilized,Health,Quantity,AdoptionSpeed
0,3,299,0,1,1,7,1,2,1,2,2,1,1,2
1,1,265,0,1,1,2,2,3,2,3,3,1,1,0
2,1,307,0,1,2,7,2,1,2,1,2,1,1,3
3,4,307,0,2,1,2,2,1,1,1,2,1,1,2
4,1,307,0,1,1,0,2,2,1,2,2,1,1,2


### Scaling

In [9]:
def scale_columns(df, columns):
    for col in columns:
        mean = df[col].mean()
        std = df[col].std()
        df[col] = (df[col] - mean) / std

if False:
    numerical_columns = ['Age', 'Breed1', 'Breed2',
                    'Gender', 'Color1', 'Color2', 'MaturitySize', 'Vaccinated',
                    'FurLength', 'Dewormed', 'Sterilized', 'Health',
                    'Quantity', 'AdoptionSpeed']
    
    scale_columns(important_df, numerical_columns)

    print("Numerical columns scaled using z-score normalization:")
    important_df.head(5)

# Model Training

In [10]:
mean_adoption_speed = important_df['AdoptionSpeed'].mean()
print(f"mean_adoption_speed: {mean_adoption_speed:.4f}")

important_df['AdoptionSpeedBinary'] = (important_df['AdoptionSpeed'] > mean_adoption_speed).astype(int)

status_counts = important_df['AdoptionSpeedBinary'].value_counts()

print("Count of rows where AdoptionSpeedBinary is 1:", status_counts.get(1, 0))
print("Count of rows where AdoptionSpeedBinary is 0:", status_counts.get(0, 0))

mean_adoption_speed: 2.5164
Count of rows where AdoptionSpeedBinary is 1: 7456
Count of rows where AdoptionSpeedBinary is 0: 7537


In [ ]:
X = important_df.drop(columns=['AdoptionSpeedBinary']).drop(columns=['AdoptionSpeed'])
y = important_df['AdoptionSpeedBinary']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(solver='liblinear'))
    ]),
    'Naive Bayes': Pipeline([
        ('model', GaussianNB())
    ]),
    'SVM': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(kernel='rbf', probability=True))
    ])
}

In [12]:
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\nCross-Validation Results on Training Data:\n")
for name, pipeline in models.items():
    print(f"Model: {name}")
    results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    for metric in scoring:
        print(f"{metric:10s}: {np.mean(results['test_' + metric]):.4f}")
    print()


Cross-Validation Results on Training Data:

Model: Logistic Regression
accuracy  : 1.0000
precision : 1.0000
recall    : 1.0000
f1        : 1.0000
roc_auc   : 1.0000

Model: Naive Bayes
accuracy  : 0.9975
precision : 0.9953
recall    : 0.9997
f1        : 0.9975
roc_auc   : 0.9998

Model: SVM
accuracy  : 0.9997
precision : 0.9998
recall    : 0.9997
f1        : 0.9997
roc_auc   : 1.0000



In [13]:
print("\nFinal Evaluation on Validation Set:\n")
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_val)
    y_proba = pipeline.predict_proba(X_val)[:, 1] if hasattr(pipeline.named_steps['model'], 'predict_proba') else None

    acc  = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred)
    rec  = recall_score(y_val, y_pred)
    f1   = f1_score(y_val, y_pred)
    roc  = roc_auc_score(y_val, y_proba) if y_proba is not None else None

    print(f"Model: {name}")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    if roc is not None:
        print(f"ROC AUC  : {roc:.4f}")
    print()


Final Evaluation on Validation Set:

Model: Logistic Regression
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1 Score : 1.0000
ROC AUC  : 1.0000

Model: Naive Bayes
Accuracy : 0.9970
Precision: 0.9940
Recall   : 1.0000
F1 Score : 0.9970
ROC AUC  : 0.9999

Model: SVM
Accuracy : 0.9997
Precision: 0.9993
Recall   : 1.0000
F1 Score : 0.9997
ROC AUC  : 1.0000



### Saving models

In [14]:
for name, pipeline in models.items():
    filename = f"{name.lower().replace(' ', '_')}_model.pkl"
    joblib.dump(pipeline, filename)
    print(f"Saved {name} to {filename}\n")

Saved Logistic Regression to logistic_regression_model.pkl

Saved Naive Bayes to naive_bayes_model.pkl

Saved SVM to svm_model.pkl



# Feature Importance

In [15]:
log_model = models['Logistic Regression'].named_steps['model']
feature_names = X_train.columns

coeffs = log_model.coef_[0]
logreg_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': coeffs,
    'Abs_Importance': np.abs(coeffs)
}).sort_values('Abs_Importance', ascending=False)

print("Top Logistic Regression Features:")
print(logreg_importance.head(20))

Top Logistic Regression Features:
          Feature  Importance  Abs_Importance
13  AdoptionSpeed   12.414111       12.414111
3          Gender    0.069390        0.069390
10     Sterilized   -0.042747        0.042747
7      Vaccinated   -0.041115        0.041115
0             Age    0.040411        0.040411
8       FurLength   -0.038841        0.038841
2          Breed2    0.030245        0.030245
12       Quantity   -0.030102        0.030102
4          Color1   -0.020876        0.020876
6    MaturitySize    0.017775        0.017775
9        Dewormed    0.013225        0.013225
1          Breed1   -0.003344        0.003344
5          Color2    0.001233        0.001233
11         Health    0.000000        0.000000


In [16]:
nb_model = models['Naive Bayes'].named_steps['model']
class_means = nb_model.theta_

mean_diff = class_means[1] - class_means[0]
nb_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': mean_diff,
    'Abs_Importance': np.abs(mean_diff)
}).sort_values('Abs_Importance', ascending=False)

print("Top Naive Bayes Features:")
print(nb_importance.head(20))

Top Naive Bayes Features:
          Feature  Importance  Abs_Importance
1          Breed1    9.021989        9.021989
0             Age    3.317511        3.317511
13  AdoptionSpeed    2.076267        2.076267
2          Breed2   -1.003173        1.003173
5          Color2   -0.149724        0.149724
4          Color1   -0.132905        0.132905
12       Quantity    0.117343        0.117343
10     Sterilized   -0.092769        0.092769
8       FurLength   -0.091278        0.091278
7      Vaccinated   -0.084414        0.084414
3          Gender    0.073656        0.073656
6    MaturitySize    0.042222        0.042222
9        Dewormed   -0.024106        0.024106
11         Health    0.000000        0.000000


In [17]:
svm_pipeline = models['SVM']
result = permutation_importance(svm_pipeline, X_val, y_val, n_repeats=10, random_state=42, scoring='accuracy')

svm_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': result.importances_mean,
    'Std': result.importances_std
}).sort_values('Importance', ascending=False)

print("Top SVM Features (via permutation importance):")
print(svm_importance.head(20))

Top SVM Features (via permutation importance):
          Feature  Importance       Std
13  AdoptionSpeed    0.501601  0.010623
5          Color2    0.000133  0.000163
12       Quantity    0.000067  0.000133
7      Vaccinated    0.000000  0.000000
9        Dewormed    0.000000  0.000000
2          Breed2    0.000000  0.000000
11         Health    0.000000  0.000000
6    MaturitySize   -0.000100  0.000153
4          Color1   -0.000100  0.000153
1          Breed1   -0.000100  0.000214
3          Gender   -0.000133  0.000221
8       FurLength   -0.000200  0.000163
0             Age   -0.000267  0.000133
10     Sterilized   -0.000300  0.000100
